In [18]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

In [19]:
raw = Path("../datasets/asl_raw/asl_alphabet_train/asl_alphabet_train")

In [20]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(raw, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(dataset.classes)

print("Classes:", num_classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes: 29
Training images: 69600
Validation images: 17400


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

In [23]:
#Freezing the pretrained fetayre layers
for param in model.features.parameters():
    param.requires_grad = False

In [26]:
# Replace the final layer for ASL classification
model.classifier[6] = nn.Linear(4096, num_classes)

In [27]:
model = model.to(device)

print("Model ready on:", device)

Model ready on: cpu


In [28]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.01
)